## Setup & evaluation metrics

This notebook loads `data/dataset.pkl` (simulated waveforms \(X \in \mathbb{R}^{N \times T \times 4}\) and parameters \(y = [R, C]\)). Features are flattened to \(\mathbb{R}^{N \times 4T}\) and standardized with `StandardScaler` (fit on train only). Train/test split: 80/20, `random_state=42`.

**Metrics recorded:** training wall time (s); **R²** on test for \(R\) and \(C\) separately (1 = perfect); **MAE** in physics units (Ω, F); **MAPE** (%) as a scale-free error proxy (**can look extreme for \(R\)** when true values are near 1 Ω — interpret next to R²); **architecture** and **complexity** (parameter counts / asymptotic notes).

In [ ]:
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
)

DATA_PATH = Path("data/dataset.pkl")
with open(DATA_PATH, "rb") as f:
    bundle = pickle.load(f)

X = np.asarray(bundle["data"])
y = np.asarray(bundle["target"])
n_samples, n_steps, n_channels = X.shape
X_flat = X.reshape(n_samples, n_steps * n_channels)

X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, random_state=42
)

x_scaler = StandardScaler()
X_train_s = x_scaler.fit_transform(X_train)
X_test_s = x_scaler.transform(X_test)

rows = []


def record(name, architecture, complexity, train_time, y_true, y_pred):
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    mae = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    mape_r = mean_absolute_percentage_error(y_true[:, 0], y_pred[:, 0])
    mape_c = mean_absolute_percentage_error(y_true[:, 1], y_pred[:, 1])
    rows.append(
        {
            "model": name,
            "train_time_s": round(float(train_time), 4),
            "R2_R": round(float(r2[0]), 4),
            "R2_C": round(float(r2[1]), 4),
            "MAE_R_ohm": round(float(mae[0]), 4),
            "MAE_C_F": float(mae[1]),
            "MAPE_R_%": round(100 * float(mape_r), 3),
            "MAPE_C_%": round(100 * float(mape_c), 3),
            "architecture": architecture,
            "complexity_notes": complexity,
        }
    )

# Linear and Ridge Regression

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

n_feat = X_train_s.shape[1]

lin = LinearRegression()
t0 = time.perf_counter()
lin.fit(X_train_s, y_train)
t_lin = time.perf_counter() - t0
pred = lin.predict(X_test_s)
p_lin = int(lin.coef_.size + lin.intercept_.size)
arch = f"LinearRegression: ŷ = X·Wᵀ + b; W shape {lin.coef_.shape}"
compl = f"{p_lin} fitted parameters; least squares (n={X_train_s.shape[0]}, d={n_feat})"
record("Linear regression", arch, compl, t_lin, y_test, pred)

ridge = Ridge(alpha=1.0, random_state=42)
t0 = time.perf_counter()
ridge.fit(X_train_s, y_train)
t_ridge = time.perf_counter() - t0
pred = ridge.predict(X_test_s)
p_r = int(ridge.coef_.size + ridge.intercept_.size)
arch = f"Ridge(alpha=1.0): same structure; W shape {ridge.coef_.shape}"
compl = f"{p_r} parameters; L2-regularized closed form"
record("Ridge (α=1)", arch, compl, t_ridge, y_test, pred)

# Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# One forest per output (R and C) avoids a single multi-output forest over-focusing on one target scale.
base_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf = MultiOutputRegressor(base_rf)
t0 = time.perf_counter()
rf.fit(X_train_s, y_train)
t_rf = time.perf_counter() - t0
pred = rf.predict(X_test_s)
T = rf.estimators_[0].n_estimators
arch = f"MultiOutputRegressor(RandomForestRegressor): {T} trees × 2 outputs (separate forests)"
compl = f"~{2 * T} total trees; training ~O(outputs × T × n log n × d) typical for tree learners"
record("Random forest", arch, compl, t_rf, y_test, pred)

# Support Vector Regression (SVR)

In [ ]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import TransformedTargetRegressor

# Scale y so R (ohms) does not dominate C (farads); metrics use original units via inverse transform.
svr = TransformedTargetRegressor(
    regressor=MultiOutputRegressor(
        SVR(kernel="rbf", C=10.0, epsilon=0.01, gamma="scale", cache_size=500)
    ),
    transformer=StandardScaler(),
)
t0 = time.perf_counter()
svr.fit(X_train_s, y_train)
t_svr = time.perf_counter() - t0
pred = svr.predict(X_test_s)
mor = svr.regressor_
n_sv = int(sum(int(e.n_support_[0]) for e in mor.estimators_))
arch = "TransformedTargetRegressor(StandardScaler) -> MultiOutputRegressor(SVR(RBF)) per target"
compl = f"~{n_sv} support vectors total; LibSVM ~O(n^2*d) worst case; joint y-scaling helps both outputs"
record("SVR (RBF)", arch, compl, t_svr, y_test, pred)

# MultiLayer Perceptron (MLP)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor

# Scale R and C jointly so the network is not dominated by the much larger R scale.
mlp = TransformedTargetRegressor(
    regressor=MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        max_iter=2000,
        early_stopping=True,
        random_state=42,
        tol=1e-4,
    ),
    transformer=StandardScaler(),
)
t0 = time.perf_counter()
mlp.fit(X_train_s, y_train)
t_mlp = time.perf_counter() - t0
pred = mlp.predict(X_test_s)
reg = mlp.regressor_
nparams = int(sum(w.size for w in reg.coefs_) + sum(b.size for b in reg.intercepts_))
arch = (
    f"TransformedTargetRegressor(StandardScaler) → MLP: {n_feat}→{reg.hidden_layer_sizes[0]}→"
    f"{reg.hidden_layer_sizes[1]}→2, {reg.activation}, {reg.solver}"
)
compl = f"{nparams} weights+biases; Adam ~O(epochs·n·params); stopped at iter {reg.n_iter_}"
record("MLP", arch, compl, t_mlp, y_test, pred)

### Summary table (run all cells above first)

Accuracy-style regression scores: higher **R²** is better (max 1); lower **MAPE** is better. Training time is wall-clock seconds on this machine and will vary.

In [ ]:
df = pd.DataFrame(rows)
# Wide columns for readability
summary = df[
    [
        "model",
        "train_time_s",
        "R2_R",
        "R2_C",
        "MAPE_R_%",
        "MAPE_C_%",
        "MAE_R_ohm",
    ]
].copy()
display(summary)
print("\nFull detail (architecture & complexity):")
display(df[["model", "architecture", "complexity_notes"]])

### Comparison: classical **Gauss–Newton** (physics-based inversion) vs ML

**`CircuitSimulator.GaussNewton`** refits \(R\) and \(C\) by minimizing \(\sum_t \|x_\mathrm{sim}(t;R,C) - x_\mathrm{meas}(t)\|^2\) using sensitivities \(\partial x/\partial R\), \(\partial x/\partial C\) — no dataset *training*, but **each test waveform** requires many repeated **Backward Euler + Newton–Raphson** simulations (expensive at inference).

Below we use the **same** random test subsample and simulation settings as in `test.py` / data generation: `amplitude=5`, `f=60` Hz, `delta_t=1e-4`, `T=0.05`, fixed initial guess \(R_0{=}1500\,\Omega\), \(C_0{=}2.5\,\mu\mathrm{F}\), `max_iter=15`. ML models are **already trained** above; we only call `.predict` on the masked inputs (cheap).

**Metrics on the subsample:** mean test **MAE** for \(R\) (Ω) and \(C\) (F); **total wall time** for all GN fits vs one batched predict per model.

If a run hits a **singular Jacobian** in the inner Newton solve, we **retry** Gauss–Newton with initial \((R,C)\) from **Ridge** on that waveform (clipped to the physical ranges) so the comparison still reflects physics-based inversion rather than abandoning hard cases.

In [ ]:
import contextlib
import io

from sklearn.metrics import mean_absolute_error, r2_score

from group_32_circuit_simulator import CircuitSimulator

amplitude = 5.0
f_hz = 60.0
delta_t = 1e-4
T_end = 0.05
x_init = np.zeros((4,))
R0, C0 = 1500.0, 2.5e-6
max_iter_gn = 15

n_test = X_test.shape[0]
X_test_3d = X_test.reshape(n_test, n_steps, n_channels)

rng_cm = np.random.default_rng(43)
n_sub = min(80, n_test)
idx = rng_cm.choice(n_test, size=n_sub, replace=False)
X_sub_s = X_test_s[idx]
y_sub = y_test[idx]

gn_buf = io.StringIO()


def run_gn(obs, R_init, C_init):
    mna = CircuitSimulator(amplitude, f_hz, R_init, C_init)
    with contextlib.redirect_stdout(gn_buf):
        return mna.GaussNewton(
            R_init, C_init, x_init, obs, delta_t, T_end, max_iter=max_iter_gn, noise=False
        )


gn_preds = []
gn_fallbacks = 0
gn_failed = 0
t_gn0 = time.perf_counter()
for k, j in enumerate(idx):
    obs = X_test_3d[j]
    try:
        R_hat, C_hat, _ = run_gn(obs, R0, C0)
    except np.linalg.LinAlgError:
        Ri, Ci = ridge.predict(X_sub_s[k : k + 1])[0]
        Ri = float(np.clip(Ri, 1.0, 2500.0))
        Ci = float(np.clip(Ci, 0.1e-6, 5e-6))
        try:
            R_hat, C_hat, _ = run_gn(obs, Ri, Ci)
            gn_fallbacks += 1
        except np.linalg.LinAlgError:
            R_hat, C_hat = np.nan, np.nan
            gn_failed += 1
    gn_preds.append([R_hat, C_hat])
t_gn = time.perf_counter() - t_gn0
gn_preds = np.asarray(gn_preds)

def subsample_mae(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    return float(mae[0]), float(mae[1])


def subsample_r2(y_true, y_pred):
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    return float(r2[0]), float(r2[1])


compare_rows = []
gn_ok = np.isfinite(gn_preds[:, 0]) & np.isfinite(gn_preds[:, 1])
if gn_ok.any():
    m_ae_r, m_ae_c = subsample_mae(y_sub[gn_ok], gn_preds[gn_ok])
    r2r, r2c = subsample_r2(y_sub[gn_ok], gn_preds[gn_ok])
else:
    m_ae_r = m_ae_c = r2r = r2c = float("nan")
compare_rows.append(
    {
        "method": "Gauss–Newton (physics)",
        "role": f"inference only; warm-start fallback: {gn_fallbacks}/{n_sub}, failed: {gn_failed}",
        "wall_time_s": round(t_gn, 4),
        "MAE_R_ohm": round(m_ae_r, 4),
        "MAE_C_F": m_ae_c,
        "R2_R": round(r2r, 4),
        "R2_C": round(r2c, 4),
        "architecture": "Iterative Gauss–Newton on nonlinear MNA + Backward Euler trajectory",
    }
)

models = [
    ("Linear regression", lin),
    ("Ridge (α=1)", ridge),
    ("Random forest", rf),
    ("SVR (RBF)", svr),
    ("MLP", mlp),
]

for name, model in models:
    t0 = time.perf_counter()
    pred_sub = model.predict(X_sub_s)
    t_pred = time.perf_counter() - t0
    m_ae_r, m_ae_c = subsample_mae(y_sub, pred_sub)
    r2r, r2c = subsample_r2(y_sub, pred_sub)
    compare_rows.append(
        {
            "method": name,
            "role": "predict on subsample (training done earlier)",
            "wall_time_s": round(t_pred, 6),
            "MAE_R_ohm": round(m_ae_r, 4),
            "MAE_C_F": m_ae_c,
            "R2_R": round(r2r, 4),
            "R2_C": round(r2c, 4),
            "architecture": "see ML summary rows above",
        }
    )

compare_df = pd.DataFrame(compare_rows)
display(
    compare_df[
        [
            "method",
            "role",
            "wall_time_s",
            "R2_R",
            "R2_C",
            "MAE_R_ohm",
        ]
    ]
)
print(
    f"Gauss–Newton subset: n={n_sub}, primary init R0={R0} Ω, C0={C0} F, max_iter={max_iter_gn}; "
    f"metrics use {int(gn_ok.sum())} converged runs ({gn_failed} failed even after Ridge warm-start)."
)
print(
    "Interpretation: GN matches the simulator (self-consistent physics); ML maps waveforms to parameters without repeatedly simulating the circuit."
)